In [ ]:
import sys
sys.path.append('..')
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from utils import merge_datasets
from wildlife_datasets.datasets.utils import load_image

In [ ]:
file_data = 'create_game_counts.csv'
folder_out = 'Game'
n_photos = 30
min_score = 0.5

root_master = '/data/wildlife_datasets/turtle_watch/TWE_flipperprofiles_registered ind_LR'
root_new = '/data/wildlife_datasets/turtle_watch'
roots_new = [
    '2017 PICS + DATASHEET',
    '2018 PICS + DATASHEET',
    '2019 PICS + DATASHEET',
    '2020 PICS + DATASHEET',
    '2021 PICS + DATASHEET',
    '2022 PICS + online database',
    '2023 PICS + online database',
    '2024 PICS + online database',
]
dataset = merge_datasets(root_new, roots_new, img_load='bbox')

In [ ]:
data = pd.read_csv(file_data)
os.makedirs(folder_out, exist_ok=True)
for identity in tqdm(data['name']):
    df_identity = dataset.df[dataset.df['identity'] == identity.lower()]
    df_identity = df_identity[df_identity['label'] == 'head']
    df_identity = df_identity[df_identity['score'] >= min_score]
    df_identity = df_identity.sort_values('score', ascending=False)
    
    idx = np.round(np.linspace(0, len(df_identity)-1, n_photos)).astype(int)
    idx = np.unique(idx)
    idx = df_identity.index[idx]

    for j, i in enumerate(idx):
        image_id = dataset.df.loc[i, 'image_id']
        dataset[i].save(f'{folder_out}/{identity}_{j:02}.webp')

In [ ]:
df = []
file_names = np.sort(os.listdir(folder_out))
for file_name in file_names:
    img = load_image(os.path.join(folder_out, file_name))
    width, height = img.size
    df.append({
        'identity': file_name.split('_')[0],
        'path': file_name,
        'orientation': '',
        'clarity': '',
        'width': width,
        'height': height,
    })
df = pd.DataFrame(df)
df.to_csv('metadata.csv')